In [1]:
!pip install -q groq pandas faiss-cpu sentence-transformers beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.1 MB/s eta 0:00:00


In [2]:
import os

# Your Groq API key (same one as before)
os.environ["GROQ_API_KEY"] = "enter_api_key"

In [3]:
from google.colab import files
from bs4 import BeautifulSoup

# Upload your 3 HTML files when the prompt appears
uploaded = files.upload()

# Extract plain text from each HTML file
documents = {}
for filename, content in uploaded.items():
    soup = BeautifulSoup(content, "html.parser")
    text = soup.get_text(separator="\n")
    # Clean up excessive blank lines
    text = "\n".join(line for line in text.splitlines() if line.strip())
    documents[filename] = text
    print(f"Extracted {len(text)} characters from {filename}")

print(f"\nLoaded {len(documents)} documents total")

Saving EStG.html to EStG.html
Saving KStG.html to KStG.html
Saving UStG.html to UStG.html
Extracted 1634980 characters from EStG.html
Extracted 369672 characters from KStG.html
Extracted 513676 characters from UStG.html

Loaded 3 documents total


In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def split_into_chunks(text, chunk_size=500, overlap=50):
    """
    Splits a long text into smaller overlapping chunks.
    Smaller chunks than before (500 vs 1000) so we can find
    more precise passages when searching.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Split all documents into chunks
all_chunks = []
for filename, text in documents.items():
    chunks = split_into_chunks(text)
    all_chunks.extend(chunks)
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Load a sentence embedding model
# This converts text into numbers so we can search by meaning
print("\nLoading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all chunks into embeddings (this takes a few minutes)
print("Embedding all chunks...")
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True)

# Build a FAISS index for fast similarity search
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

print(f"\nSearch index ready with {index.ntotal} chunks!")

EStG.html: 3634 chunks
KStG.html: 822 chunks
UStG.html: 1142 chunks

Total chunks: 5598

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all chunks...


Batches:   0%|          | 0/175 [00:00<?, ?it/s]


Search index ready with 5598 chunks!


In [8]:
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def rag_answer(question, top_k=1):
    """
    Given a question:
    1. Searches the tax law documents for the most relevant passages
    2. Passes those passages to the LLM as context
    3. Returns the LLM's answer based on that context

    top_k = how many passages to retrieve and pass to the model
    """
    # Step 1: Embed the question so we can search by meaning
    question_embedding = embedder.encode([question])

    # Step 2: Search the index for the most similar chunks
    distances, indices = index.search(np.array(question_embedding), top_k)

    # Step 3: Retrieve the actual text of the most relevant chunks
    relevant_chunks = [all_chunks[i] for i in indices[0]]
    context = "\n\n---\n\n".join(relevant_chunks)

    # Step 4: Build the prompt with context and question
    prompt = f"""You are an expert on Austrian tax law.
Use the following excerpts from Austrian tax law to answer the question.
Answer concisely and in the same language the question is asked in.

Context:
{context}

Question:
{question}

Answer:"""

    # Step 5: Send to Groq and get the answer
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=512
    )

    return response.choices[0].message.content.strip()

In [10]:
import pandas as pd
import time

# Load the test dataset
df = pd.read_csv("dataset_clean.csv")
print(f"Dataset loaded: {len(df)} rows")

results = []

print(f"\nStarting RAG inference on {len(df)} questions...\n")

for i, row in df.iterrows():
    question_id = row["id"]
    question_text = row["prompt"]

    print(f"[{i+1}/{len(df)}] Question ID: {question_id}")

    try:
        answer = rag_answer(question_text)
        print(f"  -> Answered ({len(answer)} chars)")
    except Exception as e:
        print(f"  -> ERROR: {e}")
        answer = ""

    results.append({
        "id": question_id,
        "answer": answer
    })

    # Wait between requests to stay within Groq rate limits
    time.sleep(2)

# Save output CSV
output_df = pd.DataFrame(results)
output_df.to_csv("submission_step3_rag.csv", index=False)

print(f"\nDone! Saved {len(output_df)} answers to 'submission_step3_rag.csv'")
print("\nOutput preview:")
print(output_df.head(3))

Dataset loaded: 643 rows

Starting RAG inference on 643 questions...

[1/643] Question ID: CORP-TAX-001
  -> Answered (98 chars)
[2/643] Question ID: CORP-TAX-002
  -> Answered (801 chars)
[3/643] Question ID: CORP-TAX-003
  -> Answered (272 chars)
[4/643] Question ID: CORP-TAX-004
  -> Answered (568 chars)
[5/643] Question ID: CORP-TAX-005
  -> Answered (387 chars)
[6/643] Question ID: CORP-TAX-006
  -> Answered (330 chars)
[7/643] Question ID: CORP-TAX-007
  -> Answered (631 chars)
[8/643] Question ID: CORP-TAX-008
  -> Answered (330 chars)
[9/643] Question ID: CORP-TAX-009
  -> Answered (382 chars)
[10/643] Question ID: CORP-TAX-010
  -> Answered (94 chars)
[11/643] Question ID: CORP-TAX-011
  -> Answered (166 chars)
[12/643] Question ID: CORP-TAX-012
  -> Answered (457 chars)
[13/643] Question ID: CORP-TAX-013
  -> Answered (428 chars)
[14/643] Question ID: CORP-TAX-014
  -> Answered (1003 chars)

Done! Saved 14 answers to 'submission_step3_rag.csv'

Output preview:
             id

In [11]:
import pandas as pd

# Load both files — update the filenames if they are different
first12 = pd.read_csv("first questions.csv")  # the one with questions 1-12
rest = pd.read_csv("output_model3.csv")             # the one with questions 13-643

# Combine them and reset the index
final_df = pd.concat([first12, rest]).reset_index(drop=True)

# Save as the final submission file
final_df.to_csv("submission_step3_rag.csv", index=False)

print(f"Total rows: {len(final_df)}")
print(final_df.head(3))

Total rows: 645
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Eine Körperschaft, die ein zinsloses Darlehen ...
2  CORP-TAX-003  Körperschaften, die auf Grund der Rechtsform n...
